# C-MAPSS RUL: thin orchestration notebook

**All logic lives in `src/` — this notebook only installs, imports, overrides config, calls functions, and plots.**
See `RESEARCH_PLAN.md` for scope and protocol.

## How to run (in order)
1. **Setup** — installs + mount Drive + `sys.path` + GPU check.
2. **Config** — point paths at your Drive; override any design decision here.
3. **Stage A (GPU, once per embedding config)** — Chronos-2 `embed()` → versioned cache on Drive. *Idempotent*: re-running skips if the cache key exists. Only stage needing an A100/L4 (degrades to T4 via `embed_batch_size`/`embed_dtype`).
4. **Stage B (cheap, rerunnable)** — data-fraction × loss × seed sweep + baselines on the **cached** arrays. Checkpoints every cell; a disconnect loses at most one cell. Never re-embeds.
5. **Stage C** — load result CSVs and plot the learning curves + the data-scaling curve.

The economics: **embeddings once (Stage A), everything downstream free (Stage B).**

## 1. Setup

In [ ]:
# Pinned installs. Colab already ships torch/numpy/pandas/scikit-learn/matplotlib.
# chronos-forecasting 2.x provides the official Chronos2Pipeline.embed().
%pip install -q "chronos-forecasting>=2.0.0" "coral-pytorch==1.4.0" "lightgbm>=4.0" "sktime>=1.0" "numba>=0.59" "safetensors>=0.4"

import sys, os, torch
from google.colab import drive

# Mount Drive (holds CMAPSSData/, the embedding cache, and results).
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Add the repo (kept on Drive) to sys.path so `import src.*` works.
REPO_DIR = '/content/drive/MyDrive/Predictive Maintenance LSTM/Predictive-Maintenance-LSTM'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 2. Config

The single source of truth is `src/config.py`. Override only paths (and any design decision you want to ablate) here.

In [ ]:
from src.config import Config

DRIVE = '/content/drive/MyDrive/Predictive Maintenance LSTM'
config = Config(
    dataset='FD001',
    data_dir=f'{DRIVE}/CMAPSSData',
    cache_dir=f'{DRIVE}/cache',
    results_dir=f'{DRIVE}/results',
    # --- ablation / GPU knobs (all optional) ---
    # pooling='mean',          # last_patch | mean | flatten  (changes the cache key)
    # embed_batch_size=64,     # lower for a T4
    # embed_dtype='float16',   # bfloat16 | float16 | float32
    # losses=['mse', 'corn', 'quantile'],
)
config

## 3. Stage A — Embedding pass (GPU, run once per embedding config)

Loads FD001, windows it, runs Chronos-2 `embed()` batched on GPU, and writes windows + labels + unit IDs + pooled embeddings to a Drive cache keyed by `config.embedding_cache_key()` (window size, pooling, model name, sensor set…). **Idempotent** — if the key exists it loads instead of recomputing.

In [ ]:
from src.embeddings import build_embedding_cache

cache_path = build_embedding_cache(config)   # skips instantly if the key already exists
print('embedding cache:', cache_path)
print('sidecar        :', cache_path.with_suffix('.json').read_text())

## 4. Stage B — Sweeps (cheap, rerunnable)

Data-fraction × loss × seed MLP heads on the cached embeddings, plus the baselines on the cached raw windows. Consumes the cache only — **no re-embedding**. Appends to `results/results.csv` after every cell and skips already-completed cells on restart.

In [ ]:
from src.sweep import run_sweep
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'   # heads are tiny; CPU is fine too
results_csv = run_sweep(config, device=device)
print('results CSV:', results_csv)

## 5. Stage C — Results

### Data-scaling curve (the headline figure): test error vs. training units, one line per model, error bands over seeds.

In [ ]:
import matplotlib.pyplot as plt
from src.evaluate import aggregate_data_scaling

for metric, ylabel in [('rmse', 'test RMSE'), ('nasa_score', 'NASA score')]:
    agg = aggregate_data_scaling(results_csv, metric=metric)
    plt.figure(figsize=(7, 5))
    for label, (ns, mean, std) in sorted(agg.items()):
        plt.plot(ns, mean, marker='o', label=label)
        plt.fill_between(ns, mean - std, mean + std, alpha=0.15)
    if metric == 'rmse':
        plt.axhline(14, ls='--', c='gray', lw=1, label='plan sanity check (RMSE~14)')
    plt.xscale('log'); plt.xlabel('training units'); plt.ylabel(ylabel)
    plt.title(f'Data-scaling curve (FD001) — {ylabel}')
    plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.show()

### Learning curves: validation RMSE vs. epoch, per sweep cell.

In [ ]:
from pathlib import Path
from src.evaluate import load_learning_curve

curve_files = sorted(Path(config.results_dir, 'runs', 'learning_curves').glob('*.csv'))
plt.figure(figsize=(8, 5))
for cf in curve_files:
    c = load_learning_curve(cf)
    xs, ys = c['val_rmse']
    plt.plot(xs, ys, alpha=0.6, label=cf.stem)
plt.xlabel('epoch'); plt.ylabel('val RMSE'); plt.title('Learning curves (val RMSE)')
plt.legend(fontsize=6, ncol=2); plt.grid(alpha=0.3); plt.show()